# Download OW Faults from a specific fault set via DSIS

This notebook provides a simple workflow for sending a request to DSIS using the `dsis-client` library developed by Equinor.

The following steps are covered:

1. Authenticate to DSIS using an `.env_dsis` file with the required configuration and credentials.
2. Construct and execute a query requesting faults belonging to a specific project.
3. Visualize the results returned by the query.

For more information about the required content of the `.env_dsis` file, please contact the SDD-SID team, or the DSIS team in Equinor.

In [1]:
from dotenv import load_dotenv
import os
from typing import Any
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import json
import inspect
from dsis_client import DSISClient, DSISConfig, QueryBuilder, Environment

In [2]:
load_dotenv(".env_dsis")

True

In [3]:
config = DSISConfig(
    environment=Environment.PROD,
    tenant_id=os.getenv("tenant_id"),
    client_id=os.getenv("client_id"),
    client_secret=os.getenv("client_secret"),
    access_app_id=os.getenv("resource_id"),
    dsis_username=os.getenv("dsis_function_key"),
    dsis_password=os.getenv("dsis_password"),
    subscription_key_dsauth=os.getenv("subscription_key_dsauth"),
    subscription_key_dsdata=os.getenv("subscription_key_dsdata"),
    dsis_site=os.getenv("dsis_site"),
)

In [4]:
client = DSISClient(config)
if client.test_connection():
    print("✓ Connected to DSIS API")

✓ Connected to DSIS API


In [5]:
MODEL_NAME = "OW5000"
DATABASE = "BG4FROST"
PROJECT = "VOLVE_PUBLIC"

In [11]:
# Helper function which might be incorporated in the dsis-client library in the future


def build_district_id(database: str, *, model_name: str) -> str:
    """Build DSIS district_id from database name.

    DSIS uses different district-id conventions for different models.

    Examples:
    - OpenWorksCommonModel: OpenWorksCommonModel_OW_<DB>-OW_<DB>
    - OpenWorks native models (e.g., OW5000): OpenWorks_OW_<DB>_SingleSource-OW_<DB>
    """
    if model_name == "OpenWorksCommonModel":
        return f"OpenWorksCommonModel_OW_{database}-OW_{database}"
    return f"OpenWorks_OW_{database}_SingleSource-OW_{database}"

qkw: dict = {"district_id": build_district_id(DATABASE, model_name=MODEL_NAME), "project": PROJECT}
if "model_name" in inspect.signature(QueryBuilder).parameters:
    qkw["model_name"] = MODEL_NAME

In [12]:
qkw

{'district_id': 'OpenWorks_OW_BG4FROST_SingleSource-OW_BG4FROST',
 'project': 'VOLVE_PUBLIC',
 'model_name': 'OW5000'}

In [13]:
def fetch_plane_names(client, qkw: dict) -> dict:
    """Return {fault_plane_id: fault_name} for all FaultPlane records."""
    return {
        fp["fault_plane_id"]: fp.get("fault_name", str(fp["fault_plane_id"]))
        for fp in client.execute_query(QueryBuilder(**qkw).schema("FaultPlane"), timeout=250)
    }

Fetch fault names

In [17]:
fetch_plane_names(client=client, qkw=qkw)

{'4321': 'ST10010_2013b_171013_a_update',
 '4302': 'ST10010_2013b_171013_c',
 '4303': 'ST10010_2013b_171013_d',
 '4304': 'ST10010_2013b_171013_e',
 '4320': 'ST10010_2013b_171013_f',
 '4315': 'ST10010_2013b_171013_j',
 '4316': 'ST10010_2013b_171013_k',
 '4318': 'ST10010_2013b_171013_m',
 '4308': 'ST10010_2013b_171013_n',
 '4309': 'ST10010_2013b_171013_p',
 '4307': 'ST10010_2013b_171013_q',
 '4062': 'Volve_F10_2011',
 '4063': 'Volve_F11_2011',
 '4064': 'Volve_F12_2011',
 '4072': 'Volve_F14_2011',
 '4066': 'Volve_F15_2011',
 '4073': 'Volve_F16_2011',
 '4074': 'Volve_F17_2011',
 '4075': 'Volve_F18_2011',
 '4076': 'Volve_F19_2011',
 '4053': 'Volve_F1_2011',
 '4051': 'Volve_F1_east_2011',
 '4269': 'Volve_F20_2013_update',
 '4078': 'Volve_F21_2011',
 '4079': 'Volve_F22_2011',
 '4080': 'Volve_F23_2011',
 '4081': 'Volve_F24_2011',
 '4082': 'Volve_F25_2011',
 '4083': 'Volve_F26_2011',
 '4084': 'Volve_F27_2011',
 '4085': 'Volve_F28_2011',
 '4086': 'Volve_F29_2011',
 '4054': 'Volve_F2_2011',
 '408

In [17]:
query = (
    QueryBuilder(
        model_name=MODEL_NAME,
        model_version=MODEL_VERSION,
        district_id=build_district_id(DISTRICT, model_name=MODEL_NAME),
        project=PROJECT,
    )
    .schema("FaultSegment")
)

In [19]:
for item in dsis_client.execute_query(query, timeout=250):
    print(item)

{'bounding_pt2_y': '6481067.42901409', 'bounding_pt2_x': '435107.597497127', 'bounding_pt4_y': '6481169.56827793', 'bounding_pt4_x': '434697.990355748', 'data': {'x': ['435107.59749712667', '434992.70768917905', '434872.8226721902', '434697.99035574816'], 'y': ['6481067.429014093', '6481096.077831999', '6481125.972250683', '6481169.568277932'], 'z': ['2798.537659641156', '2854.1285306370955', '2925.161310243018', '2986.1568492524516']}, 'datum_unit': 'meters', 'z_unit': 'ms', 'fault_name': 'Volve_F19_2011', 'fault_plane_version': 'UNKNOWN', 'z_domain': 'TIME', 'create_date': '2011-07-28T11:47:20.000', 'create_user_id': 'AAKA', 'native_uid': '337070', 'seismic_survey_2d_name': None, 'data_source': 'STAT', 'number_of_points': 4, 'update_date': '2014-10-16T13:22:51.000', 'bounding_pt3_x': '434697.990355748', 'datum': '0.0', 'bounding_pt1_y': '6481067.42901409', 'seismic_line_name': None, 'bounding_pt3_y': '6481169.56827793', 'bounding_pt1_x': '435107.597497127', 'fault_plane_source': 'STA